In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module=".*")

In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

%matplotlib inline

In [3]:
""" 
    Abstract:
    C–O bond activation assisted by activators such as Brønsted acids greatly improves the value of allyl alcohol in allylation;
    thus, understanding and predicting the activation energy barrier is of paramount importance. Herein, we reveal that multiple
    linear regression (MLR) analysis is a suitable tool for unifying and correlating different activators and ligands of Pd-catalyzed
    C–O bond activation of allyl alcohols. We obtain a simple model predicting activation energy barriers with different activators and 
    ligands of 393 calculated data points. Statistical tools and extensive molecular featurization have guided the development of an 
    inclusive linear regression model, providing a predictive platform and readily interpretable descriptors. It was found that easily 
    available descriptors, such as the acidity (pKa) of the activators, and the EHOMO, vertical ionization potential (VIP), and bond angle 
    (φP‑Pd‑P) of the ligands, can well describe the combined influences of steric and electronic effects, including hydrogen-bonding interactions.
    Overall, this strategy highlights the utility of MLR analysis in exploring mechanistically driven correlations across a diverse chemical 
    space in organometallic chemistry and presents an applicable workflow for C–O bond activation.
    
"""
DATA_PATH = r".\..\..\data\external\cs2c03847_si_003.xlsx"
train_df , test_df = pd.read_excel(DATA_PATH,sheet_name="Training Set"), pd.read_excel(DATA_PATH,sheet_name="Test Set")
train_df

,pKa,bond angle,E_HOMO,VIP,ΔG≠
0,(in THF),°,eV,eV,kcal/mol
1,57.974194,105.94467,-6.2878,8.4467,26.803847
2,38.393256,105.94467,-6.2878,8.4467,14.819597
3,56.262697,105.94467,-6.2878,8.4467,25.225072
4,40.968286,105.94467,-6.2878,8.4467,20.849185
...,...,...,...,...,...
389,35.957593,92.23,-5.8389,7.0065,15.556013
390,37.966598,92.23,-5.8389,7.0065,16.372785
391,41.320084,92.23,-5.8389,7.0065,19.467666
392,35.975605,92.23,-5.8389,7.0065,16.195


#### Building the Linear Regression Model using Ordinary Least Squares Method

In [ ]:
class LinearRegression:
    def __init__(self):
        self.coefficients = None
        self.intercept = None
    
    def fit(self, X:np.array, y:np.array) -> None:
        """
            calculate the coefficients and the intercept 
            of the linear regression model using the OLS method.
            X: feature matrix (numpy array or pandas DataFrame)
        """
        # Adding a column of ones to the feature matrix for the intercept
        # This is equivalent to adding a constant bias term to the model
        X = np.column_stack((np.ones(X.shape[0]), X))
        # Using moore-penrose pseudo-inverse to calculate coefficients
        # This is a more numerically stable way to compute the coefficients
        self.coefficients = np.linalg.pinv(X.T @ X) @ X.T @ y
        # The first coefficient is the intercept
        # The rest are the coefficients for the features
        self.intercept = self.coefficients[0]
        self.coefficients = self.coefficients[1:]
        
    def predict(self, X:np.array) -> np.array:
        """
            predict the target variable using the linear regression model.
            X: feature matrix (numpy array or pandas DataFrame)
        """
        X = np.column_stack((np.ones(X.shape[0]), X))
        return X @ np.concatenate(([self.intercept], self.coefficients))
        

In [32]:
linear_regression = LinearRegression()
X_train = train_df.iloc[1:, 1:-1].values.astype(float)
y_train = train_df.iloc[1:, -1].values.astype(float)
linear_regression.fit(X_train, y_train)

y_pred_train = linear_regression.predict(X_train)
assert y_pred_train.shape == y_train.shape, "The shape of the predicted values does not match the shape of the target variable."
# Display the coefficients and intercept
print("Coefficients:", linear_regression.coefficients)
print("Intercept:", linear_regression.intercept)
print("Training MSE:", mean_squared_error(y_train, y_pred_train))
print("Training MAE:", mean_absolute_error(y_train, y_pred_train))

Coefficients: [ 0.12076089 -7.31943857 -4.31618342]
Intercept: -3.6268405369544894
Training MSE: 21.971253200361865
Training MAE: 3.293130878236137


In [33]:
X_test = test_df.iloc[1:, 3:-1].values.astype(float)
y_test = test_df.iloc[1:, -1].values.astype(float)
y_pred = linear_regression.predict(X_test)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Test MSE: 41.88671107927529
Test MAE: 5.226546657537214


In [ ]:
%%time
train_df['prediction'] = np.concatenate((np.array([0]),y_pred_train),axis=0)
train_df['residual'] = np.concatenate((np.array([0]),y_train - y_pred_train),axis=0)
train_df['residual'] = train_df['residual'].abs()  

test_df['prediction'] = np.concatenate((np.array([0]),y_pred),axis=0)
test_df['residual'] = np.concatenate((np.array([0]),y_test - y_pred),axis=0)
test_df['residual'] = test_df['residual'].abs()  

with pd.ExcelWriter('./../../data/processed/predictions.xlsx') as writer:
    train_df.to_excel(writer, sheet_name='Train Set', index=False)
    test_df.to_excel(writer, sheet_name='Test Set', index=False)

CPU times: total: 78.1 ms
Wall time: 129 ms


#### Building the Linear Regression Model with numerical analysis using Gradient Descent

In [27]:
class LinearRegressionGradientDescent:
    def __init__(self):
        self.coefficients = None
        self.intercept = None
    
    def fit(self, X:np.array, y:np.array, learning_rate:float=1e-3, epochs:int=100, verbose:bool=False) -> None:
        """
            calculate the coefficients and the intercept 
            of the linear regression model using gradient descent.
            X: feature matrix (numpy array or pandas DataFrame)
        """
        # Adding a column of ones to the feature matrix for the intercept
        # This is equivalent to adding a constant bias term to the model
        X = np.column_stack((np.ones(X.shape[0]), X))
        # Initialize coefficients and intercept to zeros
        self.coefficients = np.zeros(X.shape[1])
        self.intercept = 0
        
        # Gradient descent loop
        for _ in tqdm(range(epochs),desc="Training Progress"):
            # Calculate predictions
            y_pred = X @ self.coefficients
            # Calculate gradients vector
            gradients = -2 * X.T @ (y - y_pred) / len(y)
            # Update coefficients and intercept using the gradients and learning rate
            self.coefficients -= learning_rate * gradients
            # Note: Some implementations omit the factor of 2, as it can be absorbed into the learning rate.
            self.intercept -= learning_rate * np.mean(y_pred - y)
            if _ % 25 == 0 and verbose:
                loss = np.mean((y - y_pred) ** 2)
                print(f'Epoch {_}, Loss: {loss}')
                print(f'Updated Coefficients: {self.coefficients}')
                print(f'Intercept: {self.intercept}')
                print(f'Gradient: {gradients}')
                print(f'Learning Rate: {learning_rate}')          
    
    def predict(self, X:np.array) -> np.array:
        """
            predict the target variable using the linear regression model.
            X: feature matrix (numpy array or pandas DataFrame)
        """
        X = np.column_stack((np.ones(X.shape[0]), X))
        return X @ self.coefficients

In [28]:
linear_regression = LinearRegressionGradientDescent()
X_train = train_df.iloc[1:, 1:-1].values.astype(np.float64)
y_train = train_df.iloc[1:, -1].values.astype(np.float64)
linear_regression.fit(X_train, y_train)

y_pred_train = linear_regression.predict(X_train)
assert y_pred_train.shape == y_train.shape, "The shape of the predicted values does not match the shape of the target variable."
# Display the coefficients and intercept
print("Coefficients:", linear_regression.coefficients)
print("Intercept:", linear_regression.intercept)
print("Training MSE:", mean_squared_error(y_train, y_pred_train))
print("Training MAE:", mean_absolute_error(y_train, y_pred_train))

Training Progress: 100%|██████████| 100/100 [00:00<00:00, 25942.01it/s]

Coefficients: [-1.66245967e+125 -1.66150354e+127  1.03321463e+126 -1.29024315e+126]
Intercept: -8.312298327651557e+124
Training MSE: 2.7905636279760294e+258
Training MAE: 1.6638731502390346e+129


In [29]:
X_test = test_df.iloc[1:, 3:-1].values.astype(float)
y_test = test_df.iloc[1:, -1].values.astype(float)
y_pred = linear_regression.predict(X_test)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Test MSE: 3.017661340942283e+258
Test MAE: 1.7297166541899338e+129


In [30]:
%%time
train_df['prediction'] = np.concatenate((np.array([0]),y_pred_train),axis=0)
train_df['residual'] = np.concatenate((np.array([0]),y_train - y_pred_train),axis=0)
train_df['residual'] = train_df['residual'].abs()  

test_df['prediction'] = np.concatenate((np.array([0]),y_pred),axis=0)
test_df['residual'] = np.concatenate((np.array([0]),y_test - y_pred),axis=0)
test_df['residual'] = test_df['residual'].abs()  

with pd.ExcelWriter('./../../data/processed/predictions_gd.xlsx') as writer:
    train_df.to_excel(writer, sheet_name='Train Set', index=False)
    test_df.to_excel(writer, sheet_name='Test Set', index=False)

CPU times: total: 62.5 ms
Wall time: 153 ms
